<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
    وضعیت حرکتی
</font>
</h1>

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
معرفی مجموعه داده
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
مجموعه داده آموزشی شامل 270688 سطر است که در جدول زیر، توضیحات هر ستون آمده است.
</font>
</p>

<center>
<div dir=rtl style="direction: rtl;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    
|ستون|توضیحات|
|:------:|:---:|
|timestamp|زمان ثبت داده|
|back_x|داده‌های شتاب محور X از سنسور پایین کمر|
|back_y|داده‌های شتاب محور Y از سنسور پایین کمر|
|back_z|داده‌های شتاب محور Z از سنسور پایین کمر|
|thigh_x|داده‌های شتاب محور X از حسگر روی ران|
|thigh_y|داده‌های شتاب محور Y از حسگر روی ران|
|thigh_z|داده‌های شتاب محور Z از حسگر روی ران|
|label|یک عدد صحیح نشان‌دهنده‌ی فعالیت حرکتی|
</font>
</div>
</center>


<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
 مجموعه‌داده‌ی آزمایش نیز مانند مجموعه‌ی آموزش است با این تفاوت که ستون <code>label</code> که متغیر هدف مسئله است را در خود ندارد. مجموعه‌داده‌ی آزمایش 90229 سطر و 7 ستون دارد.
</font>
</p>


<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
خواندن مجموعه داده
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    در ابتدا نیاز است تا کتابخانه‌های مورد نیاز خود را فراخوانی کنید. سپس با توجه به توضیحات بالا مجموعه‌داده‌های آموزش و آزمون را به نحو مناسبی بخوانید و پیش‌پردازش‌های لازم را روی آن‌ها انجام دهید.
    
</font>
</p>

In [22]:
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt
from scipy.fft import fft
import warnings
warnings.filterwarnings("ignore")

In [15]:
train= pd.read_csv(r"F:\Programming practices\Quera questions\MovementStatus\data\train.csv")
test= pd.read_csv(r"F:\Programming practices\Quera questions\MovementStatus\data\test.csv")

In [16]:
df= train.copy()
X_test= test.copy()

df['time']= pd.to_datetime(df['timestamp'])
X_test['time']= pd.to_datetime(X_test['timestamp'])

df.drop(columns= ['timestamp'], inplace= True)
X_test.drop(columns= ['timestamp'], inplace= True)

df['time']= (df['time'] - df['time'].iloc[0]).dt.total_seconds()
X_test['time']= (X_test['time'] - X_test['time'].iloc[0]).dt.total_seconds()

In [20]:
def butterworth_lowpass_filter(data, cutoff, fs, order= 4):
    nyquist= 0.5 * fs
    normal_cutoff= cutoff / nyquist
    b, a= butter(order, normal_cutoff, btype= 'low', analog= False)
    y= filtfilt(b, a, data)
    return y
def calculate_features(window):
   features= {}
   for col in window.columns:
        if col == 'time' or col == 'label':
            continue
        signal= window[col]
        features[f'{col}_min']= signal.min()
        features[f'{col}_max']= signal.max()
        features[f'{col}_mean']= signal.mean()
        features[f'{col}_abs_mean']= signal.abs().mean()
        features[f'{col}_rms']= np.sqrt(np.mean(signal**2))
        features[f'{col}_energy']= np.sum(signal**2)
        features[f'{col}_zcr']= np.sum(np.diff(np.sign(signal)) != 0) / (2 * len(signal))
        f= np.abs(fft(signal.to_numpy()))
        f_len= len(f) // 2
        features[f'{col}_fft_dominant_freq']= np.argmax(f[:f_len]) if f_len > 0 else 0
        features[f'{col}_fft_sum']= np.sum(f[:f_len])
        features[f'{col}_fft_mean']= np.mean(f[:f_len])
   return features

In [43]:
SAMPLING_RATE= 50
all_features= []
dl_data_windows= []
dl_labels= []
dl_subjectid= []
signal_cols= ['back_x', 'back_y', 'back_z', 'thigh_x', 'thigh_y', 'thigh_z']
for col in signal_cols:
    if len(df[col]) > 15:
        df[col]= butterworth_lowpass_filter(df[col], cutoff= 20, fs= SAMPLING_RATE)
for col in signal_cols:
    jerk_col= f'Jerk-{col} (m/s^2)'
    df[jerk_col]= np.pad(np.diff(df[col]), (1, 0), 'constant')
window_size= int(2 * SAMPLING_RATE)
step_size= int(window_size * 0.5)
for i in range(0, len(df) - window_size, step_size):
    window= df.iloc[i : i + window_size]
    unique_labels= window['label'].unique()
    if len(unique_labels) != 1:
        continue
    window_label= unique_labels[0]
    features= calculate_features(window)
    features['Activity']= window_label
    all_features.append(features)
    df_features= pd.DataFrame(all_features)

In [49]:
df_features.to_csv(r"Features.csv")

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
آموزش مدل
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    حال که داده را پاکسازی کرده‌اید، وقت آن است که مدلی آموزش دهید که بتواند متغیر هدف این مسئله را پیش‌بینی کند.
</font>
</p>

<h3 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
معیار ارزیابی
</font>
</h3>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
برای ارزیابی مدل شما از معیار `F1 Score` استفاده می‌شود و مدل میانگین‌گیری نیز به‌صورت `macro` است.
برای نمره‌گیری در این سوال مدل شما باید دارای `F1 Score` حداقل 0.40 باشد و در این حالت نمره‌ی نهایی بر اساس فرمول زیر محاسبه می‌گردد:

$$round(f1score, 3) \times 100$$
</font>
</p>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
اگر مدل شما به حدنصاب نرسد، نمره‌ی دریافتی صفر خواهد بود.
</font>
</p>

In [1]:
import pandas as pd

In [2]:
df_features= pd.read_csv(r"Features.csv")

In [3]:
from imblearn.over_sampling import SMOTE, ADASYN
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

In [12]:
# ad= ADASYN(random_state= 48)
# X, y= ad.fit_resample(df_features.drop(columns= ['Activity']), df_features['Activity'])
X= df_features.drop(columns= ['Activity'])
y= df_features['Activity']

X_train, X_val, y_train, y_val= train_test_split(X, y, random_state= 48, test_size= 0.2, stratify= y)

In [10]:
df_features['Activity']= df_features['Activity'].apply(lambda x: x-1)

In [13]:
XGB_model= XGBClassifier(n_estimators= 1000, max_depth= 10, learning_rate= 0.1, random_state= 48, verbose= 2)
XGB_model.fit(X_train, y_train)
predict= XGB_model.predict(X_val)
f1_score(y_val, predict, average= 'macro')

f:\Quera\Qvenv\lib\site-packages\xgboost\training.py:183: UserWarning: [15:17:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


0.9398994121820209

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
 پیش‌بینی برای داده تست و خروجی
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    از مدل خود برای پیش‌بینی نمونه‌های موجود در داده‌ی تست استفاده کنید و نتایج را در قالب جدولی (<code>dataframe</code>) به شکل زیر آماده کنید.
</font>
</p>

<div dir=rtl style="direction: rtl;text-align: center;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    
|ستون|توضیحات|
|------|---|
|label|نوع حرکت پیش‌بینی‌شده|
    
</font>
</div>


<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    اسم دیتافریم باید <i>submission</i> باشد؛ در غیر این صورت، سامانه داوری نمی‌تواند تلاش‌ شما را ارزیابی کند.
    <br>
    این دیتافریم تنها شامل ۱ ستون با اسم <i>label</i> است و 249 سطر دارد.
    <br>
    به ازای هر سطر موجود در دیتافریم <i>test</i> شما باید یک مقدار پیشبینی شده داشته باشید.
    <br>
    جدول زیر، ۵ سطر ابتدایی دیتافریم <code>submission</code> را نشان می‌دهد. البته در جواب شما، مقادیر ستون <i>label</i> ممکن است متفاوت باشد.
</font>
</p>

<div style="text-align: center;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    
|label|
|-----|
|1|
|2|
|1|
|8|
|8|

</font>
</div>

In [23]:
SAMPLING_RATE= 50
all_features= []
dl_data_windows= []
dl_labels= []
dl_subjectid= []
signal_cols= ['back_x', 'back_y', 'back_z', 'thigh_x', 'thigh_y', 'thigh_z']
for col in signal_cols:
    if len(X_test[col]) > 15:
        X_test[col]= butterworth_lowpass_filter(X_test[col], cutoff= 20, fs= SAMPLING_RATE)
for col in signal_cols:
    jerk_col= f'Jerk-{col} (m/s^2)'
    X_test[jerk_col]= np.pad(np.diff(X_test[col]), (1, 0), 'constant')
window_size= int(2 * SAMPLING_RATE)
step_size= int(window_size * 0.5)
for i in range(0, len(X_test) - window_size, step_size):
    window= X_test.iloc[i : i + window_size]
    unique_labels= window['label'].unique()
    if len(unique_labels) != 1:
        continue
    window_label= unique_labels[0]
    features= calculate_features(window)
    features['Activity']= window_label
    all_features.append(features)
    df_features_test= pd.DataFrame(all_features)

KeyError: 'label'

In [17]:
predict= XGB_model.predict(X_test)
submission= pd.DataFrame(predict, columns= ['label'])

ValueError: feature_names mismatch: ['Unnamed: 0', 'back_x_min', 'back_x_max', 'back_x_mean', 'back_x_abs_mean', 'back_x_rms', 'back_x_energy', 'back_x_zcr', 'back_x_fft_dominant_freq', 'back_x_fft_sum', 'back_x_fft_mean', 'back_y_min', 'back_y_max', 'back_y_mean', 'back_y_abs_mean', 'back_y_rms', 'back_y_energy', 'back_y_zcr', 'back_y_fft_dominant_freq', 'back_y_fft_sum', 'back_y_fft_mean', 'back_z_min', 'back_z_max', 'back_z_mean', 'back_z_abs_mean', 'back_z_rms', 'back_z_energy', 'back_z_zcr', 'back_z_fft_dominant_freq', 'back_z_fft_sum', 'back_z_fft_mean', 'thigh_x_min', 'thigh_x_max', 'thigh_x_mean', 'thigh_x_abs_mean', 'thigh_x_rms', 'thigh_x_energy', 'thigh_x_zcr', 'thigh_x_fft_dominant_freq', 'thigh_x_fft_sum', 'thigh_x_fft_mean', 'thigh_y_min', 'thigh_y_max', 'thigh_y_mean', 'thigh_y_abs_mean', 'thigh_y_rms', 'thigh_y_energy', 'thigh_y_zcr', 'thigh_y_fft_dominant_freq', 'thigh_y_fft_sum', 'thigh_y_fft_mean', 'thigh_z_min', 'thigh_z_max', 'thigh_z_mean', 'thigh_z_abs_mean', 'th

In [ ]:
# predict test samples
submission =

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
<b>سلول جواب‌ساز</b>
</font>
</h2>


<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    برای ساخته‌شدن فایل <code>result.zip</code> سلول زیر را اجرا کنید. توجه داشته باشید که پیش از اجرای سلول زیر تغییرات اعمال شده در نت‌بوک را ذخیره کرده باشید (<code>ctrl+s</code>) تا در صورت نیاز به پشتیبانی امکان بررسی کد شما وجود داشته باشد.
</font>
</p>

In [ ]:
import zipfile
import joblib

if not os.path.exists(os.path.join(os.getcwd(), 'movement_status.ipynb')):
    %notebook -e movement_status.ipynb


def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

submission.to_csv('submission.csv', index=False)
file_names = ['movement_status.ipynb', 'submission.csv']
compress(file_names)